In [23]:
from bs4 import BeautifulSoup
import requests

import json
import pandas as pd

import lxml.html as lh
import lxml.html.clean as lhclean

In [3]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a folder in the root directory
!mkdir -p "/content/drive/My Drive/My Folder"

Mounted at /content/drive


Data Scraping from Front Page Africa (Factual News)

In [4]:
from requests.api import request

URL='https://frontpageafricaonline.com/category/'
Categories=['news/people-innovation', 'politics', 'sports', 'county-news']
news_links=[]
Content=[]
Titles=[]
Authors=[]
Dates=[]

for ca in Categories:
  for p in range(1,15):
    x=requests.get(URL+ca+'/page/'+str(p)+'/')
    soup=BeautifulSoup(x.text, 'html.parser')
    links=soup.body.find_all(name='h2', attrs={'class':'is-title post-title'})
    for link in links:
      link=link.find('a').get('href')
      x0=requests.get(link)
      soup=BeautifulSoup(x0.text, 'html.parser')
      #Extract features of news
      i=soup.body.find(name='script', attrs={'type':'application/ld+json'})
      if i is not None:
        t=i.get_text()
        if 'headline' in t:
          #Get Content
          c=''.join(line.get_text() for line in soup.findAll('p'))
          Content.append(c)
          info=json.loads(t)
          news_links.append(info['url'])
          Titles.append(info['headline'])
          Authors.append(info['author']['name'])
          Dates.append(info['datePublished'].split('T')[0])



In [5]:
#Save in a dictionary
FPA_dic={'Title':Titles, 'Content':Content, 'Publication_date':Dates, 'Link':news_links, 'Author':Authors}

In [6]:
#Save to csv
df_FPA=pd.DataFrame.from_dict(FPA_dic)

#Convert date data format
df_FPA['Publication_date']=df_FPA['Publication_date'].astype('datetime64[ns]')
df_FPA['Publication_date']=df_FPA['Publication_date'].dt.strftime("%d/%m/%Y")

df_FPA

,Title,Content,Publication_date,Link,Author
0,CDC Announces Candlelight Vigil For Deceased P...,……MONROVIA — The Secretariat of the Mighty Coa...,24/08/2023,https://frontpageafricaonline.com/front-slider...,Press Release
1,Liberia: Pres. Weah Suspends Campaign Followin...,"……GBARNGA – President George Weah on Tuesday, ...",23/08/2023,https://frontpageafricaonline.com/front-slider...,Selma Lomax
2,Liberia: VP Howard-Taylor Inducted in the Afri...,"……The Vice President of Republic of Liberia, ...",20/09/2021,https://frontpageafricaonline.com/news/people-...,Contributing Writer
3,Liberia: Cummings Model Science and Technology...,……Paynesville – The Alexander B. Cummings Mode...,11/06/2021,https://frontpageafricaonline.com/news/tech/cu...,Press Release
4,The Race to Appoint the Successor to WTO Direc...,……At a hastily convened virtual meeting of Wor...,23/05/2020,https://frontpageafricaonline.com/news/the-rac...,Rodney Sieh
...,...,...,...,...,...
551,ZOA-Liberia Strengthens Civic Engagement for P...,"……Gbarnga, Bong County – The project officer o...",02/06/2022,https://frontpageafricaonline.com/county-news/...,Selma Lomax
552,Liberia: Single Mother of 10 Receives Cash Ass...,"……MONROVIA – Winifred Gono, a resident of Gant...",31/05/2022,https://frontpageafricaonline.com/county-news/...,Henry Karmo
553,ForumCIV Liberia and CSOs End Dialogue on Loca...,……GBARNGA – ForumCIV Liberia in partnership wi...,30/05/2022,https://frontpageafricaonline.com/county-news/...,Selma Lomax
554,Liberia: Min. McGill Endorses Eugene Kollie as...,"……DISTRICT FIVE, Bong County — The Minister of...",27/05/2022,https://frontpageafricaonline.com/politics/lib...,Selma Lomax


In [7]:
df_FPA['Label']=1
df_FPA['Publisher']='Front Page Africa'

df_FPA.to_csv('/content/drive/My Drive/My Folder/FPA_label_data.csv')

Data Scraping from ICIR

In [8]:
#Add author for df_icir
df_icir=pd.read_csv('/content/drive/My Drive/My Folder/ICIR_label_data.csv', encoding_errors='ignore')
df_icir=df_icir.dropna(axis=1)

In [10]:
df_icir

,Unnamed: 0,Content,Publication_date,Label (Fake-0| True-1),Title,Link,Author
0,0,The claim is accompanied by a short 12-second ...,14/06/2023,0,"No, video does not show ‘Sweden’s sex competit...",https://www.icirnigeria.org/no-video-does-not-...,Nurudeen AKEWUSHOLA
1,1,The claim is contained in a social media post ...,01/08/2023,0,Video does not show Nigerien minister of finan...,https://www.icirnigeria.org/video-does-not-sho...,Nurudeen AKEWUSHOLA
2,2,"A Facebook user, @Atiku Dakar, claimed that Pr...",08/07/2023,0,FACT CHECK: Did Tinubu appoint Hamza Al-Mustap...,https://www.icirnigeria.org/fact-check-did-tin...,Theophilus ADEDOKUN
3,3,The claim is circulating alongside a screen gr...,01/08/2023,0,How true is the claim that Peter Obi celebrate...,https://www.icirnigeria.org/how-true-is-the-cl...,Nurudeen AKEWUSHOLA
4,4,The images shared alongside the claim show Tru...,28/07/2023,0,8 ways to identify AI generated images,https://www.icirnigeria.org/8-ways-to-identify...,Nurudeen AKEWUSHOLA
...,...,...,...,...,...,...,...
401,401,Jonathan said this while presenting a paper at...,11/09/2017,0,"FACT CHECK: Did Jonathan, as president, ‘take ...",https://www.icirnigeria.org/fact-check-did-jon...,Kingsley OBIEJESI
402,402,INEC by saying it could not be held “direc...,19/02/2018,0,FACT CHECK: Is INEC really handicapped to curb...,https://www.icirnigeria.org/fact-check-is-inec...,Kingsley OBIEJESI
403,403,"Abubakar Nahuce, Chairman of a special panel...",03/03/2018,0,FACT CHECK: Did ‘Kano underage voting?happen i...,https://www.icirnigeria.org/fact-check-did-kan...,Kingsley OBIEJESI
404,404,"While on Sunday, he explained that what the ...",26/02/2018,0,FACT CHECK: Did Buhari promise to revitalise 1...,https://www.icirnigeria.org/fact-check-did-buh...,Kingsley OBIEJESI


In [11]:
author_names=[]
for index, row in df_icir.iterrows():
  x=requests.get(row['Link'])
  soup=BeautifulSoup(x.text, 'html.parser')
  author_name=soup.body.find(name='a',attrs={'class': 'tdb-author-name'}).get_text()
  author_names.append(author_name)

In [12]:
df_icir['Author']=author_names
df_icir

,Unnamed: 0,Content,Publication_date,Label (Fake-0| True-1),Title,Link,Author
0,0,The claim is accompanied by a short 12-second ...,14/06/2023,0,"No, video does not show ‘Sweden’s sex competit...",https://www.icirnigeria.org/no-video-does-not-...,Nurudeen AKEWUSHOLA
1,1,The claim is contained in a social media post ...,01/08/2023,0,Video does not show Nigerien minister of finan...,https://www.icirnigeria.org/video-does-not-sho...,Nurudeen AKEWUSHOLA
2,2,"A Facebook user, @Atiku Dakar, claimed that Pr...",08/07/2023,0,FACT CHECK: Did Tinubu appoint Hamza Al-Mustap...,https://www.icirnigeria.org/fact-check-did-tin...,Theophilus ADEDOKUN
3,3,The claim is circulating alongside a screen gr...,01/08/2023,0,How true is the claim that Peter Obi celebrate...,https://www.icirnigeria.org/how-true-is-the-cl...,Nurudeen AKEWUSHOLA
4,4,The images shared alongside the claim show Tru...,28/07/2023,0,8 ways to identify AI generated images,https://www.icirnigeria.org/8-ways-to-identify...,Nurudeen AKEWUSHOLA
...,...,...,...,...,...,...,...
401,401,Jonathan said this while presenting a paper at...,11/09/2017,0,"FACT CHECK: Did Jonathan, as president, ‘take ...",https://www.icirnigeria.org/fact-check-did-jon...,Kingsley OBIEJESI
402,402,INEC by saying it could not be held “direc...,19/02/2018,0,FACT CHECK: Is INEC really handicapped to curb...,https://www.icirnigeria.org/fact-check-is-inec...,Kingsley OBIEJESI
403,403,"Abubakar Nahuce, Chairman of a special panel...",03/03/2018,0,FACT CHECK: Did ‘Kano underage voting?happen i...,https://www.icirnigeria.org/fact-check-did-kan...,Kingsley OBIEJESI
404,404,"While on Sunday, he explained that what the ...",26/02/2018,0,FACT CHECK: Did Buhari promise to revitalise 1...,https://www.icirnigeria.org/fact-check-did-buh...,Kingsley OBIEJESI


In [13]:
df_icir=df_icir.rename(columns={'title':'Title', 'label':'Label (Fake-0| True-1)', 'content':'Content', 'date':'Publication_date', 'url':'Link'})

#Convert date data format
df_icir['Publication_date']=df_icir['Publication_date'].astype('datetime64[ns]')
df_icir['Publication_date']=df_icir['Publication_date'].dt.strftime("%d/%m/%Y")

#Transform labels to 0/1 in df_icir
for index, r in df_icir.iterrows():
  if r['Label (Fake-0| True-1)']==False:
    df_icir.at[index,'Label (Fake-0| True-1)']=0
  if r['Label (Fake-0| True-1)']==True:
    df_icir.at[index,'Label (Fake-0| True-1)']=1

df_icir

<ipython-input-13-5d6a3b0dcc20>:4: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  df_icir['Publication_date']=df_icir['Publication_date'].astype('datetime64[ns]')


,Unnamed: 0,Content,Publication_date,Label (Fake-0| True-1),Title,Link,Author
0,0,The claim is accompanied by a short 12-second ...,14/06/2023,0,"No, video does not show ‘Sweden’s sex competit...",https://www.icirnigeria.org/no-video-does-not-...,Nurudeen AKEWUSHOLA
1,1,The claim is contained in a social media post ...,08/01/2023,0,Video does not show Nigerien minister of finan...,https://www.icirnigeria.org/video-does-not-sho...,Nurudeen AKEWUSHOLA
2,2,"A Facebook user, @Atiku Dakar, claimed that Pr...",07/08/2023,0,FACT CHECK: Did Tinubu appoint Hamza Al-Mustap...,https://www.icirnigeria.org/fact-check-did-tin...,Theophilus ADEDOKUN
3,3,The claim is circulating alongside a screen gr...,08/01/2023,0,How true is the claim that Peter Obi celebrate...,https://www.icirnigeria.org/how-true-is-the-cl...,Nurudeen AKEWUSHOLA
4,4,The images shared alongside the claim show Tru...,28/07/2023,0,8 ways to identify AI generated images,https://www.icirnigeria.org/8-ways-to-identify...,Nurudeen AKEWUSHOLA
...,...,...,...,...,...,...,...
401,401,Jonathan said this while presenting a paper at...,09/11/2017,0,"FACT CHECK: Did Jonathan, as president, ‘take ...",https://www.icirnigeria.org/fact-check-did-jon...,Kingsley OBIEJESI
402,402,INEC by saying it could not be held “direc...,19/02/2018,0,FACT CHECK: Is INEC really handicapped to curb...,https://www.icirnigeria.org/fact-check-is-inec...,Kingsley OBIEJESI
403,403,"Abubakar Nahuce, Chairman of a special panel...",03/03/2018,0,FACT CHECK: Did ‘Kano underage voting?happen i...,https://www.icirnigeria.org/fact-check-did-kan...,Kingsley OBIEJESI
404,404,"While on Sunday, he explained that what the ...",26/02/2018,0,FACT CHECK: Did Buhari promise to revitalise 1...,https://www.icirnigeria.org/fact-check-did-buh...,Kingsley OBIEJESI


In [14]:
#Save to csv
df_icir.to_csv('/content/drive/My Drive/My Folder/ICIR_label_data.csv')

Data Scraping from GhanaFact

In [15]:
#Function
def content_extraction(page_url):
  titles=[]
  publication_dates=[]
  contents=[]
  news_links=[]
  x=requests.get(page_url)
  soup=BeautifulSoup(x.text, 'html.parser')
  links=soup.body.find_all(name='h3', attrs={'class':'elementor-post__title'})
  for link in links:
    link=link.find('a').get('href')
    news_links.append(link)
    x1=requests.get(link)
    soup1=BeautifulSoup(x1.text, 'html.parser')
    #Title
    title=soup1.body.find(name='h1', attrs={'class':'elementor-heading-title elementor-size-default'}).get_text()
    titles.append(title)
    #Date
    pd=soup1.body.find(name='span', attrs={'class':'elementor-icon-list-text elementor-post-info__item elementor-post-info__item--type-date'}).get_text()
    pd=pd.strip('/\n')
    pd=pd.strip('/\t')
    publication_dates.append(pd)
    #Content
    c=''.join(line.get_text() for line in soup1.findAll('p'))
    contents.append(c)
  #Store in a dictionary
  news_dic={'Title':titles, 'Content':contents, 'Publication_date':publication_dates, 'Link': news_links}
  return news_dic

In [16]:
from requests.api import request
URL='https://ghanafact.com/category/'
categories=['governance', 'politics', 'economy', 'health', 'environment', 'other-checks', 'deepfakes']

news_links=[]
titles=[]
publication_dates=[]
contents=[]
for c in categories:
  for page in range(1,6):
    page_link=URL+c+'/page/'+str(page)+'/'
    x=requests.get(page_link)
    soup=BeautifulSoup(x.text, 'html.parser')
    links=soup.body.find_all(name='h3', attrs={'class':'elementor-post__title'})
    for link in links:
      link=link.find('a').get('href')
      news_links.append(link)
      x1=requests.get(link)
      soup1=BeautifulSoup(x1.text, 'html.parser')
      #Title
      title=soup1.body.find(name='h1', attrs={'class':'elementor-heading-title elementor-size-default'}).get_text()
      titles.append(title)
      #Date
      pd=soup1.body.find(name='span', attrs={'class':'elementor-icon-list-text elementor-post-info__item elementor-post-info__item--type-date'}).get_text()
      pd=pd.strip('/\n')
      pd=pd.strip('/\t')
      publication_dates.append(pd)
      #Content
      content=''.join(line.get_text() for line in soup1.findAll('p'))
      contents.append(content)

In [25]:
#Store in a dictionary
news_dic={'Title':titles, 'Content':contents, 'Publication_date':publication_dates, 'Link': news_links}

#To dataframe
df_GhanaFact=pd.DataFrame(news_dic)
df_GhanaFact

,Title,Content,Publication_date,Link
0,FACT-CHECK: How much funding has gov’t provide...,Claim: 3 claims about Ghana’s CPI score and fu...,"July 12, 2023",https://ghanafact.com/2023/07/fact-check-how-m...
1,FACT-CHECK: Video showing a man punch a soldie...,Claim: A video clip shows a confrontation betw...,"March 8, 2023",https://ghanafact.com/2023/03/fact-check-video...
2,BBC Interview: Fact-Checking Prez Akufo-Addo’s...,"Claim: Claims about the Ghanaian economy, E-Le...","April 6, 2022",https://ghanafact.com/2022/04/4187/
3,FACT-CHECK: Prez Akufo-Addo falsely claims gov...,"Claim: Three claims on road construction, grow...","March 31, 2022",https://ghanafact.com/2022/03/4170/
4,FACT-CHECK: 2020 NDC manifesto did NOT propose...,Claim: 2020 NDC manifesto proposed taxing elec...,"February 23, 2022",https://ghanafact.com/2022/02/4065/
...,...,...,...,...
173,FACT-CHECK: Viral video showing cattle being w...,Claim: Video shows cattle being swept away by ...,"September 15, 2020",https://ghanafact.com/2020/09/fact-check-viral...
174,FALSE: Somali President exchanges blows with h...,Claim: Somali President exchanges blows with h...,"August 17, 2020",https://ghanafact.com/2020/08/false-somali-pre...
175,EXPLAINER: What you see online is not always w...,A photorealistic animated short video clip of ...,"June 22, 2023",https://ghanafact.com/2023/06/explainer-what-y...
176,A Brief History of Deepfakes,© 2022 All rights reserved,"June 10, 2021",https://ghanafact.com/2021/06/a-brief-history-...


In [26]:
#Publisher
df_GhanaFact['Publisher']='Ghana Fact'

In [27]:
#Save to csv
df_GhanaFact.to_csv('/content/drive/My Drive/My Folder/GhanaFact_data.csv')

Data Scraping from Dubawa

In [28]:
#Get all news and store them in dataframe
URL='https://dubawa.org/category/country/liberia/page/'
news_link=[]
news_content=[]
for page in range(1,9):
  x=requests.get(URL+str(page)+'/')
  soup=BeautifulSoup(x.text, 'html.parser')
  links=soup.body.find_all(name='h2', attrs={'class':'post-title'})
  for link in links:
    link=link.find('a').get('href')
    x1=requests.get(link)
    soup1=BeautifulSoup(x1.text, 'html.parser')
    t=soup1.body.find(name='script',attrs={'id': 'tie-schema-json'}).get_text()
    content=json.loads(t)
    news_content.append(content)

news_content_df_1=pd.DataFrame(news_content)


In [29]:
news_content_df_1

,@context,@type,dateCreated,datePublished,dateModified,headline,name,keywords,url,description,copyrightYear,articleSection,articleBody,publisher,sourceOrganization,copyrightHolder,mainEntityOfPage,author,image
0,http://schema.org,Article,2023-08-31T13:41:21+01:00,2023-08-31T13:41:21+01:00,2023-08-31T13:41:24+01:00,True! Liberia exported electricity to Ivory Co...,True! Liberia exported electricity to Ivory Co...,[],https://dubawa.org/true-liberia-exported-elect...,Claim: “Liberia exported electricity for the f...,2023,"Economy,Facebook Checks,Fact Check,Homepage,Li...",\nClaim: “Liberia exported electricity for the...,"{'@id': '#Publisher', '@type': 'Organization',...",{'@id': '#Publisher'},{'@id': '#Publisher'},"{'@type': 'WebPage', '@id': 'https://dubawa.or...","{'@type': 'Person', 'name': 'CJID Web', 'url':...","{'@type': 'ImageObject', 'url': 'https://i0.wp..."
1,http://schema.org,Article,2023-08-29T11:28:17+01:00,2023-08-29T11:28:17+01:00,2023-08-29T11:28:21+01:00,Photo of former Vice President Joseph Boakai i...,Photo of former Vice President Joseph Boakai i...,[],https://dubawa.org/photo-of-former-vice-presid...,Claim: A social media blog called Shine Liberi...,2023,"Facebook Checks,Fact Check,Liberia,Politics",\nClaim: A social media blog called Shine Libe...,"{'@id': '#Publisher', '@type': 'Organization',...",{'@id': '#Publisher'},{'@id': '#Publisher'},"{'@type': 'WebPage', '@id': 'https://dubawa.or...","{'@type': 'Person', 'name': 'CJID Web', 'url':...","{'@type': 'ImageObject', 'url': 'https://i0.wp..."
2,http://schema.org,Article,2023-08-25T16:41:25+01:00,2023-08-25T16:41:25+01:00,2023-08-25T16:41:28+01:00,DUBAWA trains Journalists in Liberia on Fact-c...,DUBAWA trains Journalists in Liberia on Fact-c...,[],https://dubawa.org/dubawa-trains-journalists-i...,"Nimba, Liberia August 21, 2023 The Fact-checki...",2023,"Homepage,Liberia,News,Newsletters &amp; Updates","\nNimba, Liberia August 21, 2023\n\n\n\nThe Fa...","{'@id': '#Publisher', '@type': 'Organization',...",{'@id': '#Publisher'},{'@id': '#Publisher'},"{'@type': 'WebPage', '@id': 'https://dubawa.or...","{'@type': 'Person', 'name': 'Caroline Anipah',...","{'@type': 'ImageObject', 'url': 'https://i0.wp..."
3,http://schema.org,Article,2023-08-10T14:10:53+01:00,2023-08-10T14:10:53+01:00,2023-08-10T14:10:57+01:00,Liberia elections glossary: 10 terms you need ...,Liberia elections glossary: 10 terms you need ...,[],https://dubawa.org/liberia-elections-glossary-...,The race to the Liberia Executive mansion has ...,2023,"Elections,Explainers,Liberia",\nThe race to the Liberia Executive mansion ha...,"{'@id': '#Publisher', '@type': 'Organization',...",{'@id': '#Publisher'},{'@id': '#Publisher'},"{'@type': 'WebPage', '@id': 'https://dubawa.or...","{'@type': 'Person', 'name': 'CJID Web', 'url':...","{'@type': 'ImageObject', 'url': 'https://i0.wp..."
4,http://schema.org,Article,2023-08-05T01:32:42+01:00,2023-08-05T01:32:42+01:00,2023-08-06T01:06:34+01:00,How Liberian politicians flout electoral laws ...,How Liberian politicians flout electoral laws ...,[],https://dubawa.org/how-liberian-politicians-fl...,"The race to Liberia’s October 10, 2023, legisl...",2023,"Explainers,Liberia","\nThe race to Liberia’s October 10, 2023, legi...","{'@id': '#Publisher', '@type': 'Organization',...",{'@id': '#Publisher'},{'@id': '#Publisher'},"{'@type': 'WebPage', '@id': 'https://dubawa.or...","{'@type': 'Person', 'name': 'CJID Web', 'url':...","{'@type': 'ImageObject', 'url': 'https://i0.wp..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,http://schema.org,Article,2021-07-28T11:43:26+01:00,2021-07-28T11:43:26+01:00,2021-07-28T11:43:31+01:00,DUBAWA’s fact-checking training holds in Liberia,DUBAWA’s fact-checking training holds in Liberia,"Featured,Liberia",https://dubawa.org/dubawas-fact-checking-train...,Twenty-two Liberian journalists drawn from the...,2021,"Featured,Liberia,Press Releases",\nTwenty-two Liberian journalists drawn from t...,"{'@id': 

In [30]:
#Feature extracted: dateCreated, datePublished, dateModified, headline, url, articleSection, articleBody, publisher

news_content_df_1=news_content_df_1[['datePublished', 'headline', 'url', 'articleSection', 'articleBody', 'author' ,'publisher']]

In [31]:
for index, r in news_content_df_1.iterrows():
  r['publisher']=r['publisher']['name']
  r['author']=r['author']['name']

news_content_df_1

,datePublished,headline,url,articleSection,articleBody,author,publisher
0,2023-08-31T13:41:21+01:00,True! Liberia exported electricity to Ivory Co...,https://dubawa.org/true-liberia-exported-elect...,"Economy,Facebook Checks,Fact Check,Homepage,Li...",\nClaim: “Liberia exported electricity for the...,CJID Web,Dubawa
1,2023-08-29T11:28:17+01:00,Photo of former Vice President Joseph Boakai i...,https://dubawa.org/photo-of-former-vice-presid...,"Facebook Checks,Fact Check,Liberia,Politics",\nClaim: A social media blog called Shine Libe...,CJID Web,Dubawa
2,2023-08-25T16:41:25+01:00,DUBAWA trains Journalists in Liberia on Fact-c...,https://dubawa.org/dubawa-trains-journalists-i...,"Homepage,Liberia,News,Newsletters &amp; Updates","\nNimba, Liberia August 21, 2023\n\n\n\nThe Fa...",Caroline Anipah,Dubawa
3,2023-08-10T14:10:53+01:00,Liberia elections glossary: 10 terms you need ...,https://dubawa.org/liberia-elections-glossary-...,"Elections,Explainers,Liberia",\nThe race to the Liberia Executive mansion ha...,CJID Web,Dubawa
4,2023-08-05T01:32:42+01:00,How Liberian politicians flout electoral laws ...,https://dubawa.org/how-liberian-politicians-fl...,"Explainers,Liberia","\nThe race to Liberia’s October 10, 2023, legi...",CJID Web,Dubawa
...,...,...,...,...,...,...,...
74,2021-07-28T11:43:26+01:00,DUBAWA’s fact-checking training holds in Liberia,https://dubawa.org/dubawas-fact-checking-train...,"Featured,Liberia,Press Releases",\nTwenty-two Liberian journalists drawn from t...,CJID Web,Dubawa
75,2021-06-28T15:34:45+01:00,Suggestions that AstraZeneca vaccine is causin...,https://dubawa.org/suggestions-that-astrazenec...,"Fact Check,Featured,Health,Liberia",\n \n\n\n\nClaim: Social media messages sugges...,Dubawa,Dubawa
76,2021-06-22T15:51:43+01:00,Debunked: Viral False Claims Following the Dem...,https://dubawa.org/debunked-viral-false-claims...,"Fact Check,Liberia,Mainstream",\n\n\n\n\nFollowing the announcement of the de...,Dubawa,Dubawa
77,2021-06-22T14:24:35+01:00,No! Gun Woman standing by Charles Taylor’s ex-...,https://dubawa.org/no-gun-woman-standing-by-ch...,"Fact Check,Featured,Liberia",\nClaim: Viral image of a gun woman is being p...,Dubawa,Dubawa


In [32]:
#Remove time from publish date
for index, r in news_content_df_1.iterrows():
  r['datePublished']=r['datePublished'].split('T')[0]

In [33]:
#Change date format
news_content_df_1['datePublished']=news_content_df_1['datePublished'].astype('datetime64[ns]')

news_content_df_1['datePublished']=news_content_df_1['datePublished'].dt.strftime("%d/%m/%Y")



<ipython-input-33-c2d222405b29>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news_content_df_1['datePublished']=news_content_df_1['datePublished'].astype('datetime64[ns]')
<ipython-input-33-c2d222405b29>:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  news_content_df_1['datePublished']=news_content_df_1['datePublished'].dt.strftime("%d/%m/%Y")


In [ ]:
news_content_df_1

In [ ]:
#Remove \n \xa0 &amp &nbsp in articleBody
news_content_df_1['articleBody']=news_content_df_1['articleBody'].replace('\xa0','',regex=True)
news_content_df_1['articleBody']=news_content_df_1['articleBody'].replace('\n','',regex=True)
for index, r in news_content_df_1.iterrows():
  doc=lh.fromstring(r['articleBody'])
  cleaner=lhclean.Cleaner(style=True)
  doc=cleaner.clean_html(doc)
  r['articleBody']=doc.text_content()


news_content_df_1

In [ ]:
#Create a label column
news_content_df_1['Label']='x'

In [ ]:
#Label fake news with 0 by checking its articleSection including 'Fact Check'
indexes=[]
for index, r in news_content_df_1.iterrows():
  if 'Fact Check' in r['articleSection']:
    r['Label']=0
    indexes.append(index)
  else:
    r['Label']=1

print('{} pieces of news have been labelled as fake news'.format(len(indexes)))
news_content_df_1

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Create a folder in the root directory
!mkdir -p "/content/drive/My Drive/My Folder"

In [ ]:
#Save labelled fake news in csv
news_content_df_1.to_csv('/content/drive/My Drive/My Folder/news_collection_Dubawa.csv')